# LANG1409 Quiz Workflow (Strict Script-Only Portfolio)

This notebook demonstrates the exact workflow family represented by the provided Python script: schema-aware synthetic data creation and script-style CSV cleaning/post-processing, with no extra inferred analytics beyond what the script supports.

Synthetic data modeled on a real analysis workflow - real data is confidential.

## Sanitization

Replaced internal references with placeholders: [LMS_BASE_URL], [LOCAL_DATA_FOLDER], [REAL_FILENAME], and [COURSE_IDENTIFIER]. No real names, identifiers, or internal URLs are used.

In [ ]:
import csv
import numpy as np
import pandas as pd

SEED = 1409
N_ROWS = 500
rng = np.random.default_rng(SEED)

FINAL_SCHEMA = ['Source.Name'] + [f'Column{i}' for i in range(1, 39)]
SYNTHETIC_FINAL_CSV = 'synthetic_lang1409_schema_matched.csv'

# Script-style intermediate file used to demonstrate cleaning behavior
RAW_EXPORT_CSV = 'synthetic_raw_canvas_export.csv'
CLEANED_AFTER_SCRIPT_CSV = 'synthetic_after_script_cleaning.csv'

## Step 1: Generate Synthetic Data (Schema-Matched)

This cell creates fully synthetic rows matching the observed sample structure and realistic value patterns (dates, scores, category mix), then introduces a few missing values so cleaning logic has work to do.

In [ ]:
tutorial_pool = [f'T{i:02d}' for i in range(1, 46)]

date_start = np.datetime64('2026-01-20T08:00:00')
date_end = np.datetime64('2026-05-12T21:00:00')
seconds_span = int((date_end - date_start) / np.timedelta64(1, 's'))

max_by_col = {
    5: 1, 7: 5, 9: 5, 11: 10, 13: 5, 15: 5, 17: 5, 19: 5,
    21: 10, 23: 5, 25: 10, 27: 10, 29: 10, 31: 5, 33: 10, 35: 0
}

rows = []
for _ in range(N_ROWS):
    t = rng.choice(tutorial_pool)
    row = {col: np.nan for col in FINAL_SCHEMA}

    row['Source.Name'] = f'{t}.csv'
    row['Column1'] = t
    row['Column2'] = float(rng.integers(69050, 69250))
    row['Column3'] = f'2025SPRING-[COURSE_IDENTIFIER]-{t}'

    sec = int(rng.integers(0, seconds_span + 1))
    dt = pd.Timestamp(date_start + np.timedelta64(sec, 's'))
    row['Column4'] = dt.strftime('%Y-%m-%d %H:%M:%S UTC')

    row['Column6'] = (
        'Scholarly Journal Articles.' if rng.random() < 0.88 else 'Wikipedia & blog posts.'
    )

    ability = float(np.clip(rng.normal(0.78, 0.16), 0.25, 0.99))
    correct = 0
    incorrect = 0

    for cidx, maxv in max_by_col.items():
        cname = f'Column{cidx}'
        if maxv == 0:
            row[cname] = 0.0
            continue
        p = float(np.clip(ability + rng.normal(0.0, 0.08), 0.05, 0.995))
        is_correct = rng.random() < p
        row[cname] = float(maxv if is_correct else 0)
        if is_correct:
            correct += 1
        else:
            incorrect += 1

    row['Column8'] = 'Synthetic answer text.'
    row['Column10'] = 'Synthetic answer text.'
    row['Column12'] = 'All of the above.'
    row['Column14'] = 'An exact phrase.'
    row['Column16'] = 'All of the above.'
    row['Column18'] = 'All of the above.'
    row['Column20'] = 'All of the above.'
    row['Column22'] = 'A, B, D'
    row['Column24'] = 'An in-text citation is usually a short reference made within the body of text.'
    row['Column26'] = 'All of the above.'
    row['Column28'] = 'All of the above.'
    row['Column30'] = 'To refine and focus your search results'
    row['Column32'] = 'Number of references'
    row['Column34'] = 'Most useful learning: search strategy and source evaluation.'

    total_score = sum(float(row[f'Column{i}']) for i in max_by_col if max_by_col[i] > 0)
    row['Column35'] = 0.0
    row['Column36'] = float(correct)
    row['Column37'] = float(incorrect)
    row['Column38'] = round((total_score / 90.0) * 100.0, 1)

    rows.append(row)

df_final = pd.DataFrame(rows, columns=FINAL_SCHEMA)

missing_idx = rng.choice(df_final.index, size=18, replace=False)
for i in missing_idx[:8]:
    df_final.loc[i, 'Column4'] = np.nan
for i in missing_idx[8:14]:
    df_final.loc[i, 'Column38'] = np.nan
for i in missing_idx[14:]:
    text_col = rng.choice(['Column6', 'Column24', 'Column34'])
    df_final.loc[i, text_col] = np.nan

df_final.to_csv(SYNTHETIC_FINAL_CSV, index=False)
print('Wrote:', SYNTHETIC_FINAL_CSV, 'shape=', df_final.shape)
df_final.head(3)

## Step 2: Recreate Script Cleaning Behavior

Your script removes a row containing section/section_id (if found) and removes the first three columns from each row. This cell demonstrates that exact behavior on a synthetic raw export file, then maps output back to the schema-matched dataset.

In [ ]:
# Build a script-like raw export with 3 leading columns and one section marker row
leading_cols = ['section', 'section_id', 'section_name']
raw_cols = leading_cols + [f'raw_{c}' for c in FINAL_SCHEMA]

df_raw = pd.DataFrame(columns=raw_cols)
df_raw['raw_Source.Name'] = df_final['Source.Name']
for c in FINAL_SCHEMA[1:]:
    df_raw[f'raw_{c}'] = df_final[c]

df_raw['section'] = 'A'
df_raw['section_id'] = '1001'
df_raw['section_name'] = 'Synthetic Section'

section_marker = {col: '' for col in raw_cols}
section_marker['section'] = 'section'
section_marker['section_id'] = 'section_id'
section_marker['section_name'] = 'section_name'

df_raw = pd.concat([pd.DataFrame([section_marker]), df_raw], ignore_index=True)
df_raw.to_csv(RAW_EXPORT_CSV, index=False)

def strip_first_row_like_script(csv_path):
    with open(csv_path, newline='', encoding='utf-8-sig') as f:
        rows = list(csv.reader(f))

    row_to_delete = None
    for i, row in enumerate(rows):
        lowered = [str(cell).lower().strip() for cell in row]
        if 'section' in lowered or 'section_id' in lowered:
            row_to_delete = i
            break

    if row_to_delete is not None:
        del rows[row_to_delete]

    rows = [row[3:] for row in rows]

    with open(CLEANED_AFTER_SCRIPT_CSV, 'w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerows(rows)

    return row_to_delete

deleted_idx = strip_first_row_like_script(RAW_EXPORT_CSV)
print('Deleted section-row index:', deleted_idx)
print('Wrote cleaned file:', CLEANED_AFTER_SCRIPT_CSV)

## Step 3: Load Cleaned Output and Run Checks

This verifies that script-style cleaning output can be standardized into the schema-matched table for downstream use, and reports any script-sample mismatches rather than guessing.

In [ ]:
df_cleaned = pd.read_csv(CLEANED_AFTER_SCRIPT_CSV)

# After script cleaning, names are not semantic in source script context; enforce known schema columns
if df_cleaned.shape[1] == len(FINAL_SCHEMA):
    df_cleaned.columns = FINAL_SCHEMA

df_cleaned['Column4'] = pd.to_datetime(df_cleaned['Column4'], format='%Y-%m-%d %H:%M:%S UTC', errors='coerce')
numeric_cols = [f'Column{i}' for i in [2,5,7,9,11,13,15,17,19,21,23,25,27,29,31,33,35,36,37,38]]
for c in numeric_cols:
    df_cleaned[c] = pd.to_numeric(df_cleaned[c], errors='coerce')

flags = []
if any(col.startswith('Column') for col in FINAL_SCHEMA[1:]):
    flags.append('Schema uses generic Column1..Column38 names; semantic headers are unavailable in provided sample.')

if df_cleaned['Column4'].isna().mean() > 0:
    flags.append('Some datetime values are missing or unparsable (expected due to injected missingness).')

print('Rows:', len(df_cleaned), '| Columns:', len(df_cleaned.columns))
print('Unique tutorials:', df_cleaned['Column1'].nunique(dropna=True))
print('Score mean:', round(df_cleaned['Column38'].mean(), 2))
print('Score median:', round(df_cleaned['Column38'].median(), 2))
print('\nMismatch / validation flags:')
for f in flags:
    print('-', f)

df_cleaned.head(3)

## Closing Notes for Workshop Planning

This strict notebook keeps only script-traceable workflow steps, so it is suitable for a defensible portfolio artifact. The synthetic dataset preserves realistic structure and quality issues without exposing confidential records. For future workshop planning, this pipeline can be reused to prepare anonymized data consistently before any separate reporting or visualization stage.